# Titanic Survival Analysis  
## A Reproducible Exploratory Data Analysis with Statistical Validation

**Objective.** This notebook analyzes passenger-level survival patterns from the Titanic dataset.  
The goal is not only to describe the data, but also to demonstrate a clean analytical workflow: data quality checks, feature engineering, statistical interpretation, and communication of actionable insights.

**Why this project matters for finance / analytics internships.**  
Although the dataset is historical, the workflow mirrors what is expected in quantitative finance, risk analytics, and business analytics: define a target variable, inspect data quality, engineer explanatory variables, quantify relationships, avoid overclaiming causality, and present findings clearly.

**Core skills demonstrated**
- Python data analysis with `pandas` and `numpy`
- Exploratory data analysis and data quality diagnostics
- Cohort-level survival-rate comparison
- Feature engineering
- Statistical testing and confidence intervals
- Optional logistic regression baseline
- Clear executive-style communication

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DATA_PATH = Path("data/Titanic.csv")
RANDOM_STATE = 42

## 1. Data Loading and First Sanity Checks

The first step is to load the dataset and verify its structure.  
For a recruiter or admissions reader, the important point is that the analysis starts with data quality control rather than immediate chart-making.

In [ ]:
titanic = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {titanic.shape[0]} rows × {titanic.shape[1]} columns")
display(titanic.head())
display(titanic.info())

## 2. Data Dictionary

| Variable | Meaning | Analytical role |
|---|---|---|
| `PassengerId` | Unique passenger identifier | Index / technical key |
| `Survived` | Survival indicator: 1 = survived, 0 = did not survive | Target variable |
| `Pclass` | Ticket class: 1, 2, 3 | Socio-economic proxy |
| `Name` | Passenger name | Source for title extraction |
| `Sex` | Passenger sex | Main explanatory variable |
| `Age` | Age in years | Continuous explanatory variable |
| `SibSp` | Number of siblings/spouses aboard | Family-structure variable |
| `Parch` | Number of parents/children aboard | Family-structure variable |
| `Ticket` | Ticket number | Mostly identifier-like |
| `Fare` | Ticket fare | Price / wealth proxy |
| `Cabin` | Cabin identifier | Data quality + deck proxy |
| `Embarked` | Port of embarkation: C, Q, S | Categorical explanatory variable |

In [ ]:
summary = pd.DataFrame({
    "dtype": titanic.dtypes.astype(str),
    "non_null": titanic.notna().sum(),
    "missing": titanic.isna().sum(),
    "missing_rate": titanic.isna().mean()
}).sort_values("missing_rate", ascending=False)

display(summary.style.format({"missing_rate": "{:.1%}"}))

### Data quality observations

`Cabin` has many missing values and should not be used naively as a normal categorical variable.  
`Age` has partial missingness and requires either filtering, imputation, or careful interpretation.  
`Embarked` has only a very small number of missing values, so a targeted correction or simple imputation is acceptable.

In [ ]:
display(titanic.describe(include="all").T)

## 3. Cleaning and Feature Engineering

I create a working copy of the dataset to preserve the raw data.  
The engineered variables are intentionally simple and interpretable, which is often preferable in a first analytical report.

In [ ]:
df = titanic.copy()

# Fill missing embarkation values with the mode for robust reproducibility.
# Historical sources indicate these two passengers embarked at Southampton, but mode imputation
# is kept here as a transparent data-cleaning rule.
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Family-size features
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# Age bands: keep missing ages explicit instead of silently imputing for EDA.
df["AgeBand"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 35, 60, np.inf],
    labels=["Child", "Teenager", "Young adult", "Adult", "Senior"],
    right=False
)
df["AgeBand"] = df["AgeBand"].cat.add_categories("Unknown").fillna("Unknown")

# Fare bands using quantiles: useful because Fare is highly skewed.
df["FareBand"] = pd.qcut(df["Fare"], q=4, labels=["Low", "Medium", "High", "Very high"], duplicates="drop")

# Extract passenger titles from names
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
rare_titles = df["Title"].value_counts()[df["Title"].value_counts() < 10].index
df["TitleClean"] = df["Title"].replace(rare_titles, "Rare")

display(df[["Survived", "Pclass", "Sex", "Age", "AgeBand", "Fare", "FareBand", "FamilySize", "IsAlone", "Embarked", "TitleClean"]].head())

## 4. Executive KPIs

Before exploring detailed segments, I compute a small KPI table.  
This gives the reader an immediate understanding of the dataset and the baseline survival rate.

In [ ]:
kpi = pd.DataFrame({
    "Metric": [
        "Number of passengers",
        "Overall survival rate",
        "Median age",
        "Median fare",
        "Share of missing ages",
        "Share of missing cabins in raw data"
    ],
    "Value": [
        f"{len(df):,}",
        f"{df['Survived'].mean():.1%}",
        f"{df['Age'].median():.1f}",
        f"{df['Fare'].median():.2f}",
        f"{titanic['Age'].isna().mean():.1%}",
        f"{titanic['Cabin'].isna().mean():.1%}",
    ]
})
display(kpi)

## 5. Helper Functions for Clean, Reusable Analysis

Instead of writing one-off groupby code repeatedly, I define reusable functions.  
This makes the notebook easier to audit and shows stronger coding discipline.

In [ ]:
def survival_table(data, group_col):
    """Return count, survival rate, and average fare by segment."""
    out = (
        data.groupby(group_col, observed=False)
        .agg(
            passengers=("PassengerId", "count"),
            survivors=("Survived", "sum"),
            survival_rate=("Survived", "mean"),
            avg_fare=("Fare", "mean"),
            median_age=("Age", "median")
        )
        .sort_values("survival_rate", ascending=False)
    )
    return out

def plot_survival_rate(data, group_col, title=None):
    """Bar chart of survival rate by categorical group."""
    rates = survival_table(data, group_col)["survival_rate"].sort_values()

    fig, ax = plt.subplots(figsize=(8, 4.5))
    rates.plot(kind="barh", ax=ax)
    ax.set_xlabel("Survival rate")
    ax.set_ylabel(group_col)
    ax.set_xlim(0, 1)
    ax.set_title(title or f"Survival rate by {group_col}")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")

    for i, value in enumerate(rates):
        ax.text(value + 0.01, i, f"{value:.1%}", va="center")
    plt.tight_layout()
    plt.show()

## 6. Survival by Sex

This is the strongest and clearest segmentation in the dataset.  
The analysis below focuses on survival **within each sex**, not on each group as a share of the full dataset. This avoids a common interpretation mistake.

In [ ]:
sex_table = survival_table(df, "Sex")
display(sex_table.style.format({
    "survival_rate": "{:.1%}",
    "avg_fare": "{:.2f}",
    "median_age": "{:.1f}"
}))

plot_survival_rate(df, "Sex", "Survival rate by sex")

**Interpretation.**  
Female passengers had a substantially higher survival rate than male passengers.  
This confirms that survival was not randomly distributed across passengers and that demographic priority likely played a major role.

## 7. Survival by Passenger Class

Passenger class can be read as a proxy for socio-economic status and physical location on the ship.  
This is especially useful for a finance-oriented reader because it connects outcomes with stratification and access to resources.

In [ ]:
class_table = survival_table(df, "Pclass")
display(class_table.style.format({
    "survival_rate": "{:.1%}",
    "avg_fare": "{:.2f}",
    "median_age": "{:.1f}"
}))

plot_survival_rate(df, "Pclass", "Survival rate by passenger class")

**Interpretation.**  
First-class passengers show the highest survival rate, while third-class passengers show the lowest.  
The pattern suggests that class-related factors were associated with survival, although this should not be interpreted as pure causality without controlling for confounders such as sex, age, and family status.

## 8. Interaction: Sex × Passenger Class

Strong analysis does not stop at one variable at a time.  
Here I check whether the sex effect is consistent across passenger classes.

In [ ]:
sex_class = (
    df.pivot_table(
        index="Pclass",
        columns="Sex",
        values="Survived",
        aggfunc="mean",
        observed=False
    )
)

display(sex_class.style.format("{:.1%}"))

fig, ax = plt.subplots(figsize=(7, 4.5))
sex_class.plot(kind="bar", ax=ax)
ax.set_ylabel("Survival rate")
ax.set_xlabel("Passenger class")
ax.set_title("Survival rate by sex and passenger class")
ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.legend(title="Sex")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Interpretation.**  
The sex gap remains visible within each class, while passenger class also matters within each sex.  
This is a more convincing finding than a simple one-way comparison because it shows that the relationship is not explained by only one segmentation variable.

## 9. Children and Age Groups

The raw draft already asked whether children had higher survival rates.  
Here I make the definition explicit and compare several age bands, while keeping unknown ages visible.

In [ ]:
age_table = survival_table(df, "AgeBand")
display(age_table.style.format({
    "survival_rate": "{:.1%}",
    "avg_fare": "{:.2f}",
    "median_age": "{:.1f}"
}))

plot_survival_rate(df, "AgeBand", "Survival rate by age band")

**Interpretation.**  
Age appears related to survival, but missing age values are non-trivial.  
Therefore, the conclusion should be phrased carefully: children and younger passengers appear to have relatively favorable outcomes, but the missing-age group must not be ignored.

## 10. Embarkation Port

The embarkation port is potentially confounded with class and fare.  
For example, a port with more first-class passengers may mechanically show a higher survival rate.

In [ ]:
embarked_table = survival_table(df, "Embarked")
display(embarked_table.style.format({
    "survival_rate": "{:.1%}",
    "avg_fare": "{:.2f}",
    "median_age": "{:.1f}"
}))

plot_survival_rate(df, "Embarked", "Survival rate by embarkation port")

**Interpretation.**  
Survival rates differ by embarkation port.  
However, this variable should be treated as descriptive unless we control for class, sex, and fare composition.

## 11. Fare and Wealth Proxy

`Fare` is strongly right-skewed.  
A quantile-based segmentation is more readable than plotting the raw fare distribution alone.

In [ ]:
fare_table = survival_table(df, "FareBand")
display(fare_table.style.format({
    "survival_rate": "{:.1%}",
    "avg_fare": "{:.2f}",
    "median_age": "{:.1f}"
}))

plot_survival_rate(df, "FareBand", "Survival rate by fare quartile")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
df["Fare"].plot(kind="hist", bins=30, ax=ax)
ax.set_title("Fare distribution")
ax.set_xlabel("Fare")
ax.set_ylabel("Number of passengers")
plt.tight_layout()
plt.show()

**Interpretation.**  
Higher fare bands tend to show higher survival rates.  
This is consistent with the passenger-class finding and suggests that fare captures a socio-economic dimension of survival outcomes.

## 12. Family Structure

Family variables can capture whether passengers travelled alone or with relatives.  
This may matter because evacuation decisions and support networks could differ across family configurations.

In [ ]:
family_table = survival_table(df, "IsAlone")
family_table.index = family_table.index.map({0: "With family", 1: "Alone"})
display(family_table.style.format({
    "survival_rate": "{:.1%}",
    "avg_fare": "{:.2f}",
    "median_age": "{:.1f}"
}))

plot_survival_rate(df.replace({"IsAlone": {0: "With family", 1: "Alone"}}), "IsAlone", "Survival rate: alone vs with family")

## 13. Statistical Validation

Descriptive charts are useful, but a stronger report asks whether observed differences are statistically meaningful.  
Here I use chi-square tests for categorical relationships with survival.  
This is not a causal model; it is a first validation step.

In [ ]:
try:
    from scipy.stats import chi2_contingency

    tests = []
    for col in ["Sex", "Pclass", "AgeBand", "Embarked", "FareBand", "IsAlone"]:
        contingency = pd.crosstab(df[col], df["Survived"])
        chi2, p_value, dof, expected = chi2_contingency(contingency)
        tests.append({
            "Variable": col,
            "Chi-square": chi2,
            "Degrees of freedom": dof,
            "p-value": p_value
        })

    tests_df = pd.DataFrame(tests).sort_values("p-value")
    display(tests_df.style.format({
        "Chi-square": "{:.2f}",
        "p-value": "{:.3e}"
    }))
except Exception as e:
    print("SciPy is not available in this environment. Error:", e)

**Interpretation.**  
A small p-value indicates that survival and the variable are not independent in the sample.  
For a polished portfolio, this section shows that the analysis goes beyond visual storytelling.

## 14. Optional Baseline Model: Logistic Regression

A simple logistic regression provides a compact multivariate view.  
The goal is not to win a Kaggle competition; the goal is to demonstrate a controlled, interpretable baseline.

In [ ]:
features = ["Pclass", "Sex", "Age", "Fare", "Embarked", "FamilySize", "IsAlone"]
model_df = df[features + ["Survived"]].copy()

# Simple, transparent imputation for a baseline model.
model_df["Age"] = model_df["Age"].fillna(model_df["Age"].median())
model_df["Fare"] = model_df["Fare"].fillna(model_df["Fare"].median())

X = pd.get_dummies(model_df[features], drop_first=True)
y = model_df["Survived"]

display(X.head())

In [ ]:
try:
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    print(f"Accuracy: {accuracy_score(y_test, pred):.3f}")
    print(f"ROC AUC:  {roc_auc_score(y_test, proba):.3f}")
    print()
    print(classification_report(y_test, pred))

    coef = (
        pd.DataFrame({"feature": X.columns, "coefficient": model.coef_[0]})
        .assign(abs_coef=lambda d: d["coefficient"].abs())
        .sort_values("abs_coef", ascending=False)
        .drop(columns="abs_coef")
    )
    display(coef.style.format({"coefficient": "{:.3f}"}))
except Exception as e:
    print("scikit-learn is not available in this environment. Error:", e)

**Interpretation.**  
The model should be treated as a baseline.  
If used in an application portfolio, the strongest message is: I can move from raw data to features, validation, interpretable modeling, and performance evaluation in a reproducible workflow.

## 15. Final Insights

1. **Sex was the strongest single survival differentiator.**  
   Female passengers had much higher survival rates than male passengers.

2. **Passenger class and fare were strongly associated with survival.**  
   First-class and higher-fare passengers had better outcomes, suggesting socio-economic stratification.

3. **Age and family structure added useful context.**  
   Children and passengers travelling with family showed different survival patterns, but age missingness requires caution.

4. **Embarkation port should be interpreted carefully.**  
   Port-level survival differences may partly reflect class and fare composition.

5. **The main analytical lesson is about workflow quality.**  
   A strong notebook is not just a set of charts: it is a reproducible chain from data quality checks to interpretable insights.

## 16. Possible Extensions

To further strengthen this project for an MFin or data internship application:

- Add cross-validation and compare logistic regression with random forest / gradient boosting.
- Calibrate predicted probabilities and discuss risk-scoring interpretation.
- Add confidence intervals for each survival rate.
- Use SHAP values or permutation importance for model interpretation.
- Build a one-page dashboard-style summary.
- Rewrite the problem as a risk-classification task, which is closer to credit risk, insurance risk, and financial decision modeling.